# DM-AHR-ViT — one notebook, all models

This notebook runs the project's own phase scripts end to end and **stops after the trained models exist**. It does not run evaluation, XAI, statistics, report generation, or the external real-hair validation.

What it produces, under `ARTIFACT_ROOT/checkpoints/`:

- `dmahr/` — repaired base diffusion generator (50,000 iterations)
- `dmahr_v2/` — fine-tuned diffusion generator (+20,000 pasted-real-strand iterations)
- `vit/<arm>_seed<seed>/best.pt` — ViT-B/16 classifiers for arms A, A*, B and Cv2

Runtime: the full study is many GPU-hours (about 70,000 diffusion iterations plus 32 ViT runs on the published 8-seed design). An A100 (Colab Pro) is strongly recommended. A free T4 will work but is much slower. Everything is written to Google Drive, so a disconnected session resumes instead of restarting: every phase script skips or reloads work that already exists.

Set `FAST_MODE = True` in the config cell for a quick 3-seed smoke of the whole chain.


## 1. Config

Edit `FAST_MODE` if you want the shorter pre-registered 3-seed design. Leave it `False` for the published 8-seed design. The diffusion budgets and Cv2 sampler settings are the project's production values; do not change them if you want the recorded recipe.


In [ ]:
# ---- editable config ----
FAST_MODE = False                       # True -> 3 seeds (42/123/2026) instead of 8
DIFF_BASE_ITERS = 50000                 # repaired base generator
DIFF_FINETUNE_ITERS = 20000             # real-strand fine-tune from the base
CV2_STEPS = 50                          # strided DDIM steps
CV2_START_T = 600                       # partial-noise start index of 2000
CV2_SHARDS = 4
SEEDS_FULL = [42, 123, 2026, 7, 11, 17, 23, 31]
SEEDS_PRE = [42, 123, 2026]
SEEDS = SEEDS_PRE if FAST_MODE else SEEDS_FULL
ARMS = ["A", "Astar", "B", "Cv2"]
REPO_URL = "https://github.com/Icecubesaad/ML-shit.git"
PIN_COMMIT = "c439d59"  # pinned to the commit that produced the current paper
print("FAST_MODE", FAST_MODE, "| seeds", SEEDS, "| diffusion", DIFF_BASE_ITERS, "+", DIFF_FINETUNE_ITERS)


## 2. Get the code and put data on Drive

`/content` is wiped when a Colab session ends, and the diffusion checkpoints plus 11,720 source images are large. Both data and artifacts therefore live under `MyDrive/dm_ahr_vit/`. Re-running any cell continues from whatever is already there.


In [ ]:
import os, subprocess, sys, time
from pathlib import Path

IN_COLAB = Path("/content").exists()
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/dm_ahr_vit")
else:
    BASE = Path.cwd() / "_dm_ahr_run"
BASE.mkdir(parents=True, exist_ok=True)
DATA_ROOT = BASE / "data"
ARTIFACT_ROOT = BASE / "artifacts"
DATA_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["DATA_ROOT"] = str(DATA_ROOT)
os.environ["ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("DRIVE BASE:", BASE)
print("DATA_ROOT:", DATA_ROOT)
print("ARTIFACT_ROOT:", ARTIFACT_ROOT)


In [ ]:
# Clone the pinned repository unless we are already inside it.
if not Path("configs/protocol.json").exists():
    if Path("ML-shit").exists():
        os.chdir("ML-shit")
    else:
        subprocess.run(["git", "clone", REPO_URL, "ML-shit"], check=True)
        os.chdir("ML-shit")
subprocess.run(["git", "checkout", PIN_COMMIT], check=True)
print("repo at", Path.cwd(), "commit", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip())
assert Path("third_party/dmahr").exists(), "vendored upstream DM-AHR code is missing"


## 3. Dependencies and GPU check

Pinned versions from `requirements.txt`. On Colab this replaces the preinstalled torch on purpose: the project's sampler was verified with these exact versions.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| device", name)
if "T4" in name:
    print("WARNING: T4 is supported but roughly 3-5x slower than the A100 used for the recorded run.")
assert torch.cuda.is_available(), "No GPU detected. Colab: Runtime -> Change runtime type -> GPU."


In [ ]:
# Every phase cell goes through this wrapper so paths and environment stay consistent.
def run(script, *args, check=True):
    cmd = [sys.executable, f"scripts/{script}", *[str(a) for a in args]]
    print("RUN", " ".join(cmd), flush=True)
    t0 = time.time()
    p = subprocess.run(cmd, env=os.environ.copy())
    print(f"EXIT {p.returncode} in {time.time()-t0:.1f}s", flush=True)
    if check and p.returncode != 0:
        raise RuntimeError(f"{script} failed with exit code {p.returncode}")
    return p.returncode


## 4. Phase 1 — download the data

All 11,720 images of ISIC collection 212 plus HAM10000 metadata and lesion segmentations. Public, no account or key. Resumable: downloaded files are cached on Drive.


In [ ]:
run("00_download_data.py")
import json
rep = json.loads((ARTIFACT_ROOT / "logs" / "download_report.json").read_text())
print("downloaded:", rep["isic"])


## 5. Phase 2 — audit, deduplicate and freeze the lesion-grouped split

Recomputes every count from the files, merges confirmed near-duplicate links into lesion groups, and writes the frozen 70/15/15 split. A fresh download reproduces the reference split hash; the leakage test must pass before training anything.


In [ ]:
run("01_audit_and_splits.py")
audit = json.loads((ARTIFACT_ROOT / "logs" / "data_audit.json").read_text())
print("cohort:", audit.get("frozen_splits", {}).get("n_images"))
print("split hash:", audit.get("frozen_splits", {}).get("splits_sha256"))


## 6. Phase 3 — hair screening, Arm B and synthetic pairs

The detector thresholds are tuned on train/validation only and frozen before any test use. This phase writes the DullRazor images (Arm B), the 128 px control (A*) and the synthetic hairy/hairless pairs used to train the diffusion model.


In [ ]:
run("02_hair_pipeline.py")


## 7. Phase 4 — harvest hair strands for the fine-tune

The 20,000-iteration fine-tune trains on pasted real strands taken from train/validation images only. This CPU phase builds `hair_layers/` and never reads the test split.


In [ ]:
run("40_harvest_hair.py", "--workers", 8)


In [ ]:
# quick check without importing pandas
n_layers = len(list((DATA_ROOT / "hair_layers").glob("*.png")))
print("hair layers:", n_layers)
assert n_layers > 0, "no hair strands harvested; the fine-tune cannot run"


## 8. Phase 5 — train the repaired base diffusion generator

50,000 iterations on the synthetic pairs, selecting on validation. Checkpoints are written as it goes, so an interrupted session resumes from `train_state.pt`.


In [ ]:
run("04_dmahr_train.py", "--iterations", DIFF_BASE_ITERS)
base_dir = ARTIFACT_ROOT / "checkpoints" / "dmahr"
base_ckpts = sorted(base_dir.glob("gen_iter*.pth"),
                    key=lambda p: int(p.stem.split("_")[-1]))
base_ckpt = base_ckpts[-1]
print("base checkpoint:", base_ckpt, base_ckpt.stat().st_size)


## 9. Phase 6 — fine-tune for 20,000 iterations on pasted real strands

Starts from the base generator above and writes into a separate `dmahr_v2/` directory. The recorded run never overwrites `dmahr/`.


In [ ]:
run("04_dmahr_train.py", "--composite",
    "--warm-start", base_ckpt,
    "--out-name", "dmahr_v2",
    "--iterations", DIFF_FINETUNE_ITERS,
    "--num-workers", 8)
ft_ckpt = ARTIFACT_ROOT / "checkpoints" / "dmahr_v2" / "final_gen.pth"
assert ft_ckpt.exists(), f"missing fine-tuned checkpoint {ft_ckpt}"
print("fine-tuned checkpoint:", ft_ckpt, ft_ckpt.stat().st_size)


## 10. Phase 7 — generate the Cv2 images and fill hair-absent passthrough

Production sampler: 128 px, partial-noise start at index 600 of 2,000, 50 strided DDIM steps, FP32. Shards are resumable because already-written images are skipped. The held-out synthetic test pairs are processed separately so paired image-quality metrics remain possible downstream (this notebook does not compute them).


In [ ]:
for shard in range(CV2_SHARDS):
    run("05_dmahr_sample.py", "--shard", shard, "--n-shards", CV2_SHARDS,
        "--arm-name", "Cv2", "--checkpoint", ft_ckpt,
        "--respaced", "--steps", CV2_STEPS, "--start-t", CV2_START_T)
print("Cv2 hair-present images:", len(list((DATA_ROOT / "processed" / "Cv2").glob("*.png"))))


In [ ]:
run("05_dmahr_sample.py", "--checkpoint", ft_ckpt, "--pairs-split", "test",
    "--arm-name", "Cv2", "--respaced", "--steps", CV2_STEPS, "--start-t", CV2_START_T)
run("05b_fill_c_passthrough.py", "--arm-name", "Cv2")
print("Cv2 complete:", len(list((DATA_ROOT / "processed" / "Cv2").glob("*.png"))))


## 11. Phase 8 — train the ViT-B/16 classifiers

Same matched recipe for every arm: ImageNet-pretrained ViT-B/16, saved split, same augmentation and schedule. Each run resumes from its own directory. This is the last training phase.


In [ ]:
# sanity run first
run("06_train_vit.py", "--arm", "A", "--seed", SEEDS[0])


In [ ]:
for arm in ARMS:
    for seed in SEEDS:
        if arm == "A" and seed == SEEDS[0]:
            continue  # already trained above
        run("06_train_vit.py", "--arm", arm, "--seed", seed)
print("all ViT runs complete")


## 12. Models produced

That is the whole job. The cells below only list what was created; nothing else is run.


In [ ]:
import hashlib
ck = ARTIFACT_ROOT / "checkpoints"
diffusion = sorted((ck / "dmahr").glob("gen_iter*.pth")) + sorted((ck / "dmahr_v2").glob("*.pth"))
vit = sorted(ck.glob("vit/*/best.pt"))
print("diffusion generators:", len(diffusion))
for p in diffusion:
    print(" ", p.relative_to(ck), f"{p.stat().st_size/1e6:.1f} MB")
print("ViT checkpoints:", len(vit))
for p in vit:
    print(" ", p.relative_to(ck), f"{p.stat().st_size/1e6:.1f} MB")
print("\nAll models are under:", ck)
